# XFLR5 Baseline vs Optimised run comparison

In [1]:
from pathlib import Path
import pandas as pd

# Standalone setup
ROOT = Path('/Users/gherardi/Documents/GitHub/glider_optimization/artifacts/xfoil')
OUT_COMPARISON = ROOT / 'Results' / 'xflr5_baseline_vs_optimised_grid.csv'
OUT_CONV_SUMMARY = ROOT / 'Results' / 'xflr5_convergence_summary.csv'
EXPECTED_ALPHA_COUNT = 121  # alpha in [-30, 30] with step 0.5

if not OUT_COMPARISON.exists():
    raise FileNotFoundError(
        f'Comparison CSV not found: {OUT_COMPARISON}\n'
        'Run compare_xfoil_runs.ipynb first to generate it.'
    )

comparison = pd.read_csv(OUT_COMPARISON)

# Convergence summary (% over expected 121 alpha points)
summary = (
    comparison.groupby('Re', as_index=False)
    .agg(
        baseline_converged=('baseline_has_data', 'sum'),
        optimised_converged=('optimised_has_data', 'sum')
    )
)
summary['expected_points'] = EXPECTED_ALPHA_COUNT
summary['baseline_convergence_pct'] = 100.0 * summary['baseline_converged'] / summary['expected_points']
summary['optimised_convergence_pct'] = 100.0 * summary['optimised_converged'] / summary['expected_points']

summary.to_csv(OUT_CONV_SUMMARY, index=False)
print('Loaded comparison CSV:', OUT_COMPARISON)
print('Saved convergence summary CSV:', OUT_CONV_SUMMARY)
summary

Loaded comparison CSV: /Users/gherardi/Documents/GitHub/glider_optimization/artifacts/xfoil/Results/xflr5_baseline_vs_optimised_grid.csv
Saved convergence summary CSV: /Users/gherardi/Documents/GitHub/glider_optimization/artifacts/xfoil/Results/xflr5_convergence_summary.csv


,Re,baseline_converged,optimised_converged,expected_points,baseline_convergence_pct,optimised_convergence_pct
0,160,0,0,121,0.000000,0.000000
1,641,121,120,121,100.000000,99.173554
2,1597,121,120,121,100.000000,99.173554
3,3020,120,121,121,99.173554,100.000000
4,4896,121,120,121,100.000000,99.173554
5,7206,120,119,121,99.173554,98.347107
6,9930,117,118,121,96.694215,97.520661
7,13039,120,120,121,99.173554,99.173554
8,16506,120,116,121,99.173554,95.867769
9,20295,112,111,121,92.561983,91.735537


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
from pathlib import Path

# Load comparison CSV (created by previous cells)
csv_path = Path('/Users/gherardi/Documents/GitHub/glider_optimization/artifacts/xfoil/Results/xflr5_baseline_vs_optimised_grid.csv')
if not csv_path.exists():
    raise FileNotFoundError(f'Comparison CSV not found: {csv_path}')

df_cmp = pd.read_csv(csv_path)
re_values = sorted(df_cmp['Re'].dropna().astype(int).unique().tolist())

if not re_values:
    raise ValueError('No Reynolds values found in comparison CSV.')

# Consistent colors across all subplots
BASELINE_COLOR = 'tab:blue'
OPTIMISED_COLOR = 'tab:orange'

def _add_missing_markers(ax, x_missing, y_ref, color, marker, label):
    if len(x_missing) == 0:
        return
    y0, y1 = ax.get_ylim()
    if np.isfinite(y0) and np.isfinite(y1):
        y_mark = y0 + 0.05 * (y1 - y0)
    else:
        finite_ref = y_ref[np.isfinite(y_ref)]
        y_mark = float(finite_ref.min()) if len(finite_ref) > 0 else 0.0
    ax.scatter(
        x_missing,
        np.full_like(x_missing, y_mark, dtype=float),
        marker=marker,
        s=28,
        color=color,
        alpha=0.9,
        label=label,
        zorder=4,
    )

def _safe_ratio(cl, cd):
    out = np.full_like(cl, np.nan, dtype=float)
    valid = np.isfinite(cl) & np.isfinite(cd) & (np.abs(cd) > 1e-12)
    out[valid] = cl[valid] / cd[valid]
    return out

def plot_for_re(selected_re):
    sub = df_cmp[df_cmp['Re'] == int(selected_re)].copy().sort_values('alpha')

    alpha = sub['alpha'].to_numpy(dtype=float)

    b_cl = sub['baseline_CL'].to_numpy(dtype=float)
    o_cl = sub['optimised_CL'].to_numpy(dtype=float)
    b_cd = sub['baseline_CD'].to_numpy(dtype=float)
    o_cd = sub['optimised_CD'].to_numpy(dtype=float)

    b_clcd = _safe_ratio(b_cl, b_cd)
    o_clcd = _safe_ratio(o_cl, o_cd)

    # Missing locations (NaN) -> line cuts + explicit markers
    miss_b_cl = alpha[np.isnan(b_cl)]
    miss_o_cl = alpha[np.isnan(o_cl)]
    miss_b_cd = alpha[np.isnan(b_cd)]
    miss_o_cd = alpha[np.isnan(o_cd)]
    miss_b_clcd = alpha[np.isnan(b_clcd)]
    miss_o_clcd = alpha[np.isnan(o_clcd)]

    fig, axes = plt.subplots(3, 1, figsize=(10, 11), sharex=True)

    # --- CL plot ---
    axes[0].plot(alpha, b_cl, label='Baseline CL', color=BASELINE_COLOR, linewidth=2)
    axes[0].plot(alpha, o_cl, label='Optimised CL', color=OPTIMISED_COLOR, linewidth=2)
    axes[0].set_ylabel('CL')
    axes[0].set_title(f'Baseline vs Optimised at Re={int(selected_re)}')
    axes[0].grid(True, alpha=0.3)
    _add_missing_markers(axes[0], miss_b_cl, b_cl, BASELINE_COLOR, 'x', 'Baseline CL missing')
    _add_missing_markers(axes[0], miss_o_cl, o_cl, OPTIMISED_COLOR, '+', 'Optimised CL missing')
    axes[0].legend(loc='best')

    # --- CD plot ---
    axes[1].plot(alpha, b_cd, label='Baseline CD', color=BASELINE_COLOR, linewidth=2)
    axes[1].plot(alpha, o_cd, label='Optimised CD', color=OPTIMISED_COLOR, linewidth=2)
    axes[1].set_ylabel('CD')
    axes[1].grid(True, alpha=0.3)
    _add_missing_markers(axes[1], miss_b_cd, b_cd, BASELINE_COLOR, 'x', 'Baseline CD missing')
    _add_missing_markers(axes[1], miss_o_cd, o_cd, OPTIMISED_COLOR, '+', 'Optimised CD missing')
    axes[1].legend(loc='best')

    # --- CL/CD plot ---
    axes[2].plot(alpha, b_clcd, label='Baseline CL/CD', color=BASELINE_COLOR, linewidth=2)
    axes[2].plot(alpha, o_clcd, label='Optimised CL/CD', color=OPTIMISED_COLOR, linewidth=2)
    axes[2].set_xlabel('alpha (deg)')
    axes[2].set_ylabel('CL/CD')
    axes[2].grid(True, alpha=0.3)
    _add_missing_markers(axes[2], miss_b_clcd, b_clcd, BASELINE_COLOR, 'x', 'Baseline CL/CD missing')
    _add_missing_markers(axes[2], miss_o_clcd, o_clcd, OPTIMISED_COLOR, '+', 'Optimised CL/CD missing')
    axes[2].legend(loc='best')

    plt.tight_layout()
    plt.show()

    print(f'Baseline converged points: {np.sum(~np.isnan(b_cl))}/{len(alpha)}')
    print(f'Optimised converged points: {np.sum(~np.isnan(o_cl))}/{len(alpha)}')

re_dropdown = widgets.Dropdown(
    options=re_values,
    value=re_values[0],
    description='Re:',
    layout=widgets.Layout(width='280px')
)

ui = widgets.interactive_output(plot_for_re, {'selected_re': re_dropdown})
display(re_dropdown, ui)

Dropdown(description='Re:', layout=Layout(width='280px'), options=(160, 641, 1597, 3020, 4896, 7206, 9930, 130…

Output()